## Baseline Modeling & Tree-Based Models

In [43]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [44]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from src.data_utils import load_processed

In [45]:
df=load_processed('model_ready.csv')

### Train-Validation Split

In [46]:
X = df.drop(columns=['TARGET'])
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### Logistic Regression Baseline

In [47]:
num_col = X_train.select_dtypes(include = 'number').columns
cat_col = X_train.select_dtypes(include = 'object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_col),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_col)
    ]
)

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, n_jobs=-1))
])

In [48]:
pipe.fit(X_train, y_train)

y_pred = pipe.predict_proba(X_test)[:,1]

lgr_auc = roc_auc_score(y_test, y_pred)

print(f'Logistic Regression AUC: {lgr_auc}')


Logistic Regression AUC: 0.7586362274076947


#### Results

Validation AUC: 0.7586

### Decision Tree Model

In [39]:
pipe_tree = Pipeline([
    ('preprocessor', preprocessor_tree),
    ('model', DecisionTreeClassifier(max_depth=8, random_state=42))
])

In [40]:
pipe_tree.fit(X_train, y_train)

y_pred_tr = pipe_tree.predict_proba(X_test)[:,1]

tree_auc = roc_auc_score(y_test, y_pred_tr)

print(f'Decision Tree AUC: {tree_auc}')

Decision Tree AUC: 0.7130730547701396


In [41]:
pipe_tree.fit(X_train, y_train)

y_train_pred = pipe_tree.predict_proba(X_train)[:, 1]
y_valid_pred = pipe_tree.predict_proba(X_test)[:, 1]

train_auc = roc_auc_score(y_train, y_train_pred)
valid_auc = roc_auc_score(y_test, y_valid_pred)

print(f"Decision Tree Train AUC: {train_auc:.4f}")
print(f"Decision Tree Validation AUC: {valid_auc:.4f}")

Decision Tree Train AUC: 0.7369
Decision Tree Validation AUC: 0.7131


#### Results

Train AUC: 0.7369  
Validation AUC: 0.7131  

Small gap indicates controlled variance.